In [9]:
import os
from pathlib import Path
from dotenv import load_dotenv

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

env_path = repo_root / ".env"
load_dotenv(env_path, override=True)

print("CWD:", cwd)
print("Repo root:", repo_root)
print("ENV path:", env_path)
print("ENV exists:", env_path.exists())

print("EVENT_HUB_NAMESPACE =", os.getenv("EVENT_HUB_NAMESPACE"))
print("EVENT_HUB_ORDERS_NAME =", os.getenv("EVENT_HUB_ORDERS_NAME"))
print("EVENT_HUB_DELIVERY_NAME =", os.getenv("EVENT_HUB_DELIVERY_NAME") or os.getenv("EVENT_HUB_COURIERS_NAME"))
print("EVENT_HUB_ORDERS_LISTEN_CONNECTION_STRING loaded =", bool(os.getenv("EVENT_HUB_ORDERS_LISTEN_CONNECTION_STRING")))
print("EVENT_HUB_DELIVERY_LISTEN_CONNECTION_STRING loaded =", bool(os.getenv("EVENT_HUB_DELIVERY_LISTEN_CONNECTION_STRING")))

CWD: /workspaces/Stream_Analytics-Group_7/notebooks
Repo root: /workspaces/Stream_Analytics-Group_7
ENV path: /workspaces/Stream_Analytics-Group_7/.env
ENV exists: True
EVENT_HUB_NAMESPACE = iesstsabbadbaa-grp-06-10
EVENT_HUB_ORDERS_NAME = group_7_orders
EVENT_HUB_DELIVERY_NAME = group_7_delivery
EVENT_HUB_ORDERS_LISTEN_CONNECTION_STRING loaded = True
EVENT_HUB_DELIVERY_LISTEN_CONNECTION_STRING loaded = True


In [10]:
import json

order_schema_path = repo_root / "schemas" / "order_events_v1.avsc"
delivery_schema_path = repo_root / "schemas" / "courier_events_v1.avsc"

print("order schema path:", order_schema_path)
print("delivery schema path:", delivery_schema_path)
print("order schema exists:", order_schema_path.exists())
print("delivery schema exists:", delivery_schema_path.exists())

with open(order_schema_path, "r", encoding="utf-8") as f:
    order_schema = json.dumps(json.load(f))

with open(delivery_schema_path, "r", encoding="utf-8") as f:
    delivery_schema = json.dumps(json.load(f))

print("Schemas loaded OK")

order schema path: /workspaces/Stream_Analytics-Group_7/schemas/order_events_v1.avsc
delivery schema path: /workspaces/Stream_Analytics-Group_7/schemas/courier_events_v1.avsc
order schema exists: True
delivery schema exists: True
Schemas loaded OK


In [11]:
import os
import findspark

os.environ["SPARK_HOME"] = "/workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)
print("SPARK_HOME:", os.environ["SPARK_HOME"])
print("JAVA_HOME:", os.environ["JAVA_HOME"])

PySpark version: 4.0.1
SPARK_HOME: /workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [12]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("group7-eventhub-bronze")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
        "org.apache.spark:spark-avro_2.13:4.0.1"
    )
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.1


In [13]:
event_hub_namespace = os.getenv("EVENT_HUB_NAMESPACE")

orders_hub = os.getenv("EVENT_HUB_ORDERS_NAME")
delivery_hub = os.getenv("EVENT_HUB_DELIVERY_NAME") or os.getenv("EVENT_HUB_COURIERS_NAME")

orders_connection_str = os.getenv("EVENT_HUB_ORDERS_LISTEN_CONNECTION_STRING")
delivery_connection_str = os.getenv("EVENT_HUB_DELIVERY_LISTEN_CONNECTION_STRING")

print("Orders listen loaded:", bool(orders_connection_str))
print("Delivery listen loaded:", bool(delivery_connection_str))
print("Namespace:", event_hub_namespace)
print("Orders hub:", orders_hub)
print("Delivery hub:", delivery_hub)

Orders listen loaded: True
Delivery listen loaded: True
Namespace: iesstsabbadbaa-grp-06-10
Orders hub: group_7_orders
Delivery hub: group_7_delivery


In [14]:
orders_kafka_options = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{orders_connection_str}";',
    "subscribe": orders_hub,
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

delivery_kafka_options = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{delivery_connection_str}";',
    "subscribe": delivery_hub,
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

print("Kafka options ready")

Kafka options ready


In [15]:
orders_batch = (
    spark.read
    .format("kafka")
    .options(**orders_kafka_options)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

delivery_batch = (
    spark.read
    .format("kafka")
    .options(**delivery_kafka_options)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

print("orders batch count:", orders_batch.count())
print("delivery batch count:", delivery_batch.count())

orders_batch.select("topic", "partition", "offset", "timestamp").show(10, truncate=False)
delivery_batch.select("topic", "partition", "offset", "timestamp").show(10, truncate=False)

orders batch count: 415


delivery batch count: 1107


+--------------+---------+------+-----------------------+
|topic         |partition|offset|timestamp              |
+--------------+---------+------+-----------------------+
|group_7_orders|0        |0     |2026-04-16 20:22:44.267|
|group_7_orders|0        |1     |2026-04-16 20:22:44.33 |
|group_7_orders|0        |2     |2026-04-16 20:22:44.33 |
|group_7_orders|0        |3     |2026-04-16 20:22:44.33 |
|group_7_orders|0        |4     |2026-04-16 20:22:44.439|
|group_7_orders|0        |5     |2026-04-16 20:22:44.533|
|group_7_orders|0        |6     |2026-04-16 20:22:44.643|
|group_7_orders|0        |7     |2026-04-16 20:22:44.737|
|group_7_orders|0        |8     |2026-04-16 20:22:46.049|
|group_7_orders|0        |9     |2026-04-16 20:22:46.159|
+--------------+---------+------+-----------------------+
only showing top 10 rows
+----------------+---------+------+-----------------------+
|topic           |partition|offset|timestamp              |
+----------------+---------+------+--------

In [16]:
orders_raw = (
    spark.readStream
    .format("kafka")
    .options(**orders_kafka_options)
    .load()
)

delivery_raw = (
    spark.readStream
    .format("kafka")
    .options(**delivery_kafka_options)
    .load()
)

print("Raw streams created")

Raw streams created


In [17]:
orders_raw.printSchema()
delivery_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [18]:
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.functions import col

In [19]:
orders_parsed = (
    orders_raw
    .select(from_avro(col("value"), order_schema).alias("data"))
    .select("data.*")
)

delivery_parsed = (
    delivery_raw
    .select(from_avro(col("value"), delivery_schema).alias("data"))
    .select("data.*")
)

print("AVRO parsing ready")

AVRO parsing ready


In [20]:
orders_parsed.printSchema()
delivery_parsed.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- order_value: double (nullable = true)
 |-- is_promo: boolean (nullable = true)
 |-- estimated_delivery_minutes: integer (nullable = true)
 |-- version: integer (nullable = true)

root
 |-- event_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- courier_status: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- version: integer (nullable = true)



In [21]:
orders_bronze = (
    orders_parsed
    .withColumn("event_time", col("event_ts"))
    .withColumn("ingest_time", col("ingest_ts"))
)

delivery_bronze = (
    delivery_parsed
    .withColumn("event_time", col("event_ts"))
    .withColumn("ingest_time", col("ingest_ts"))
)

print("Bronze DataFrames ready")

Bronze DataFrames ready


In [22]:
from pathlib import Path

bronze_orders_path = repo_root / "data" / "bronze" / "orders"
bronze_delivery_path = repo_root / "data" / "bronze" / "delivery"

check_orders_path = repo_root / "data" / "checkpoints" / "orders_bronze"
check_delivery_path = repo_root / "data" / "checkpoints" / "delivery_bronze"

bronze_orders_path.mkdir(parents=True, exist_ok=True)
bronze_delivery_path.mkdir(parents=True, exist_ok=True)
check_orders_path.mkdir(parents=True, exist_ok=True)
check_delivery_path.mkdir(parents=True, exist_ok=True)

print("bronze_orders_path =", bronze_orders_path)
print("bronze_delivery_path =", bronze_delivery_path)

bronze_orders_path = /workspaces/Stream_Analytics-Group_7/data/bronze/orders
bronze_delivery_path = /workspaces/Stream_Analytics-Group_7/data/bronze/delivery


In [23]:
import shutil

shutil.rmtree(str(check_orders_path), ignore_errors=True)
shutil.rmtree(str(check_delivery_path), ignore_errors=True)

print("Old checkpoints deleted")

Old checkpoints deleted


In [24]:
orders_query = (
    orders_bronze.writeStream
    .format("parquet")
    .option("path", str(bronze_orders_path))
    .option("checkpointLocation", str(check_orders_path))
    .outputMode("append")
    .start()
)

delivery_query = (
    delivery_bronze.writeStream
    .format("parquet")
    .option("path", str(bronze_delivery_path))
    .option("checkpointLocation", str(check_delivery_path))
    .outputMode("append")
    .start()
)

print("Bronze queries started")
import time
time.sleep(30)
print("Wait finished")
orders_query.stop()
delivery_query.stop()

print("Bronze queries stopped")

26/04/19 17:25:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/19 17:25:48 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Bronze queries started
Wait finished
Bronze queries stopped


26/04/19 17:26:18 WARN DAGScheduler: Failed to cancel job group f1c7ff2f-cf45-49d4-916b-7a8b096105c5. Cannot find active jobs for it.
26/04/19 17:26:18 WARN DAGScheduler: Failed to cancel job group f1c7ff2f-cf45-49d4-916b-7a8b096105c5. Cannot find active jobs for it.
26/04/19 17:26:18 WARN DAGScheduler: Failed to cancel job group 2671a039-75d1-49c4-81f2-34d61716cef7. Cannot find active jobs for it.
26/04/19 17:26:18 WARN DAGScheduler: Failed to cancel job group 2671a039-75d1-49c4-81f2-34d61716cef7. Cannot find active jobs for it.


In [25]:
import os

print("Orders bronze files:")
print(os.listdir(bronze_orders_path))

print("\nDelivery bronze files:")
print(os.listdir(bronze_delivery_path))

Orders bronze files:
['part-00003-98b27160-7378-4090-88f6-063cf1d1c7ea-c000.snappy.parquet', '.part-00000-badd2bb2-5e2f-4b69-864f-0a5a0b6326b1-c000.snappy.parquet.crc', 'part-00000-badd2bb2-5e2f-4b69-864f-0a5a0b6326b1-c000.snappy.parquet', 'part-00002-5b0751ec-7848-4191-99a8-f744ca025a1f-c000.snappy.parquet', '.part-00003-98b27160-7378-4090-88f6-063cf1d1c7ea-c000.snappy.parquet.crc', 'part-00001-d7db74a2-9de7-4543-a92f-94782a7b86ef-c000.snappy.parquet', '_spark_metadata', '.part-00002-5b0751ec-7848-4191-99a8-f744ca025a1f-c000.snappy.parquet.crc', '.part-00001-d7db74a2-9de7-4543-a92f-94782a7b86ef-c000.snappy.parquet.crc']

Delivery bronze files:
['part-00001-4ac30217-0552-443d-affa-1ea720f3f5aa-c000.snappy.parquet', '.part-00000-6558200d-cfc3-4d5d-af4a-35a30e0e79ad-c000.snappy.parquet.crc', '.part-00001-4ac30217-0552-443d-affa-1ea720f3f5aa-c000.snappy.parquet.crc', 'part-00000-6558200d-cfc3-4d5d-af4a-35a30e0e79ad-c000.snappy.parquet', '_spark_metadata']


In [26]:
orders_bronze_df = spark.read.parquet(str(bronze_orders_path))
delivery_bronze_df = spark.read.parquet(str(bronze_delivery_path))

print("Orders bronze count:", orders_bronze_df.count())
print("Delivery bronze count:", delivery_bronze_df.count())

orders_bronze_df.show(5, truncate=False)
delivery_bronze_df.show(5, truncate=False)

Orders bronze count: 94
Delivery bronze count: 738
+------------------------------------+------------------------------------+----------------+-----------------------+-----------------------+-------------+-------+----------+-----------+--------+--------------------------+-------+-----------------------+-----------------------+
|event_id                            |order_id                            |event_type      |event_ts               |ingest_ts              |restaurant_id|zone_id|courier_id|order_value|is_promo|estimated_delivery_minutes|version|event_time             |ingest_time            |
+------------------------------------+------------------------------------+----------------+-----------------------+-----------------------+-------------+-------+----------+-----------+--------+--------------------------+-------+-----------------------+-----------------------+
|64dce1a6-2081-43d3-a08e-37e1a9b9aa4d|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|prep_started    |2026-04-16 20:19:24.093